In [1]:
%pip install -U datasets

%pip install --upgrade pip
%pip install --disable-pip-version-check \
    torch \
    torchdata

%pip install \
    transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [10]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [12]:


dataset = load_dataset("json", data_files="/content/bit_dialogues.jsonl", split="train")


Generating train split: 0 examples [00:00, ? examples/s]

In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [32]:
def tokenize(example):
    text = example["prompt"] + tokenizer.eos_token + example["response"] + tokenizer.eos_token
    tokens = tokenizer(text, truncation=True, max_length=128, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

In [33]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

model = AutoModelForCausalLM.from_pretrained("distilgpt2")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [34]:
training_args = TrainingArguments(
    output_dir="./bit_model",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10,
    save_total_limit=1,
    logging_steps=5
)



In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)



In [36]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,5.764200
10,5.179400
15,4.585700
20,4.486700


TrainOutput(global_step=21, training_loss=4.980246861775716, metrics={'train_runtime': 64.1, 'train_samples_per_second': 0.608, 'train_steps_per_second': 0.328, 'total_flos': 1273821659136.0, 'train_loss': 4.980246861775716, 'epoch': 3.0})

In [52]:
model.save_pretrained("bitChatbot_model")
tokenizer.save_pretrained("bitChatbot_model")

('bitChatbot_model/tokenizer_config.json',
 'bitChatbot_model/special_tokens_map.json',
 'bitChatbot_model/vocab.json',
 'bitChatbot_model/merges.txt',
 'bitChatbot_model/added_tokens.json',
 'bitChatbot_model/tokenizer.json')

In [54]:
!zip -r bitChatbot_model.zip bitChatbot_model/

  adding: bitChatbot_model/ (stored 0%)
  adding: bitChatbot_model/vocab.json (deflated 59%)
  adding: bitChatbot_model/merges.txt (deflated 53%)
  adding: bitChatbot_model/special_tokens_map.json (deflated 60%)
  adding: bitChatbot_model/tokenizer.json (deflated 82%)
  adding: bitChatbot_model/model.safetensors (deflated 7%)
  adding: bitChatbot_model/tokenizer_config.json (deflated 54%)
  adding: bitChatbot_model/config.json (deflated 52%)
  adding: bitChatbot_model/generation_config.json (deflated 24%)


In [55]:
from google.colab import files
files.download("bitChatbot_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>